<h3 style="color: red;">CREATE DB</h3>

In [1]:
import sqlite3
from pathlib import Path

ROOT_DIR = ROOT_DIR = Path.cwd() # Current Working Directory (Dir atual)
DB_NAME = 'omdb.sqlite3'
DB_FILE = ROOT_DIR / DB_NAME
TABLE_NAME = 'movies'

if __name__ == '__main__': 
    connection = sqlite3.connect(DB_FILE)
    cursor = connection.cursor()

<h3 style="color: red;">CREATE TABLE</h3>

In [53]:
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    year TEXT,
    released TEXT,
    runtime TEXT,
    genre TEXT,
    director TEXT,
    rating TEXT DEFAULT "Unrated"
)
""")
connection.commit()

<h3 style="color: red;">ADD MOVIE</h3>

In [57]:
import requests
import sqlite3
from pathlib import Path

apikey = "9a51a7ae"
movie = "Scream 3"    # CHANGE THE MOVIE HERE!!!

url = f"https://www.omdbapi.com/?apikey={apikey}&t={movie}"

response = requests.get(url)
data = response.json()

ROOT_DIR = Path.cwd()
DB_FILE = ROOT_DIR / "omdb.sqlite3"

connection = sqlite3.connect(DB_FILE)
cursor = connection.cursor()

if data["Response"] == "True":

    title = data["Title"]
    year = data["Year"]
    released = data["Released"]
    runtime = data["Runtime"]
    genre = data["Genre"]
    director = data["Director"]

    cursor.execute("SELECT * FROM movies WHERE title = ?", (title,))
    exists = cursor.fetchone()

    if not exists:

        cursor.execute("""
            INSERT INTO movies
            (title, year, released, runtime, genre, director)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (title, year, released, runtime, genre, director))

        connection.commit()

        print("Movie added!")

    else:
        print("Movie already exists.")

else:
    print("Error:", data["Error"])

cursor.close()
connection.close()

Movie added!


<h3 style="color: red;">SHOW DB</h3>

In [78]:
connection = sqlite3.connect(DB_FILE)
cursor = connection.cursor()

cursor.execute(f'SELECT * FROM {TABLE_NAME}') 


rows = cursor.fetchall() 
if not rows:
    print('No data found!')
else:
    for row in rows: 
        print(row)


cursor.close()
connection.close()


(1, 'Scream', '1996', '20 Dec 1996', '111 min', 'Horror, Mystery', 'Wes Craven', '8')
(2, 'Scream 2', '1997', '12 Dec 1997', '120 min', 'Horror, Mystery', 'Wes Craven', 'Unrated')
(3, 'Scream 3', '2000', '04 Feb 2000', '116 min', 'Horror, Mystery', 'Wes Craven', 'Unrated')


<h3 style="color: red;">CREATE EXCEL</h3>

In [79]:
connection = sqlite3.connect(DB_FILE)
cursor = connection.cursor()
EXCEL_FILE = Path.cwd() / "rating.xlsx"

import pandas as pd
df = pd.read_sql_query("SELECT * FROM movies", connection)

with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl', mode='w') as writer:
    df.to_excel(writer, index=False) # sobrescrever ou criar caso não exista

print(df)
print('xlms created/updated!')

   id     title  year     released  runtime            genre    director  \
0   1    Scream  1996  20 Dec 1996  111 min  Horror, Mystery  Wes Craven   
1   2  Scream 2  1997  12 Dec 1997  120 min  Horror, Mystery  Wes Craven   
2   3  Scream 3  2000  04 Feb 2000  116 min  Horror, Mystery  Wes Craven   

    rating  
0        8  
1  Unrated  
2  Unrated  
xlms created/updated!


<h3 style="color: red;">OPEN EXCEL</h3>

In [83]:
import os
from pathlib import Path

EXCEL_FILE = Path.cwd() / "rating.xlsx" 

if EXCEL_FILE.exists():
    os.startfile(EXCEL_FILE) 
    print("Excel opened!")
else:
    print("File does not exist!")

Excel opened!


<h3 style="color: red;">EXCEL WITHOUT RATING (XLWINGS)</h3>

In [ ]:
# abre o Excel e aplique o filtro automaticamente, mantendo todas as macros e formatos,

import xlwings as xw   # pip install xlwings #pip install pywin32 xlwings
from pathlib import Path

EXCEL_FILE = Path.cwd() / "rating.xlsx"


wb = xw.Book(EXCEL_FILE) # abrir o Excel existente

sheet = wb.sheets[0]
sheet.api.AutoFilter(1, "Unrated")  # 1 = segunda coluna (B), ajuste se necessário

print("Excel opened with only Unrated movies visible!")

XlwingsError: Make sure to have "pywin32", a dependency of xlwings, installed.

<h3 style="color: red;">* DROP TABLE</h3>

In [52]:
cursor.execute("DROP TABLE movies")
connection.commit()